# 02 - Data Preprocessing

In this notebook, we will prepare the CIC-IDS2017 dataset for machine learning.

The preprocessing workflow includes:

- Loading and inspecting the raw datasets.
- Cleaning column names and labels.
- Combining the datasets.
- Handling data quality issues.
- Investigating duplicate and invalid records.
- Preparing features and target labels.
- Splitting the data and applying transformations.

**Important:** The original raw datasets will never be modified.

In [8]:
import pandas as pd
import numpy as np
import os
import glob

from IPython.display import display

In [9]:
DATA_PATH = "../dataset/raw"

files = glob.glob(os.path.join(DATA_PATH, "*.csv"))

print("Number of CSV files:", len(files))

for file in files:
    print(os.path.basename(file))

Number of CSV files: 8
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [10]:
raw_datasets = {}

for file in files:

    file_name = os.path.basename(file)

    df = pd.read_csv(file, low_memory=False)

    raw_datasets[file_name] = df

    print(f"{file_name} loaded successfully!")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv loaded successfully!
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv loaded successfully!
Friday-WorkingHours-Morning.pcap_ISCX.csv loaded successfully!
Monday-WorkingHours.pcap_ISCX.csv loaded successfully!
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv loaded successfully!
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv loaded successfully!
Tuesday-WorkingHours.pcap_ISCX.csv loaded successfully!
Wednesday-workingHours.pcap_ISCX.csv loaded successfully!


In [11]:
total_rows = sum(
    len(df) for df in raw_datasets.values()
)

print("Number of datasets:", len(raw_datasets))
print("Total rows:", total_rows)

Number of datasets: 8
Total rows: 2830743


In [12]:
total_rows = sum(
    len(df) for df in raw_datasets.values()
)

print("Number of datasets:", len(raw_datasets))
print("Total rows:", total_rows)

Number of datasets: 8
Total rows: 2830743


In [13]:
working_datasets = {}

for name, df in raw_datasets.items():

    temp_df = df.copy()

    # Remove leading and trailing spaces
    temp_df.columns = temp_df.columns.str.strip()

    working_datasets[name] = temp_df

    print(f"{name}: {len(temp_df.columns)} columns")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 79 columns
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 79 columns
Friday-WorkingHours-Morning.pcap_ISCX.csv: 79 columns
Monday-WorkingHours.pcap_ISCX.csv: 79 columns
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 79 columns
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 79 columns
Tuesday-WorkingHours.pcap_ISCX.csv: 79 columns
Wednesday-workingHours.pcap_ISCX.csv: 79 columns


In [14]:
for name, df in working_datasets.items():

    print(f"\nDataset: {name}")
    print("Label column exists:", "Label" in df.columns)
    print("Number of columns:", len(df.columns))


Dataset: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Friday-WorkingHours-Morning.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Monday-WorkingHours.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Tuesday-WorkingHours.pcap_ISCX.csv
Label column exists: True
Number of columns: 79

Dataset: Wednesday-workingHours.pcap_ISCX.csv
Label column exists: True
Number of columns: 79


In [15]:
label_mapping = {
    "Web Attack � Brute Force": "Web Attack - Brute Force",
    "Web Attack � XSS": "Web Attack - XSS",
    "Web Attack � Sql Injection": "Web Attack - SQL Injection"
}

for name, df in working_datasets.items():

    df["Label"] = df["Label"].replace(label_mapping)

    print(f"{name}: Labels normalized successfully!")

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: Labels normalized successfully!
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: Labels normalized successfully!
Friday-WorkingHours-Morning.pcap_ISCX.csv: Labels normalized successfully!
Monday-WorkingHours.pcap_ISCX.csv: Labels normalized successfully!
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: Labels normalized successfully!
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: Labels normalized successfully!
Tuesday-WorkingHours.pcap_ISCX.csv: Labels normalized successfully!
Wednesday-workingHours.pcap_ISCX.csv: Labels normalized successfully!


In [16]:
for name, df in working_datasets.items():

    suspicious_labels = [
        label for label in df["Label"].unique()
        if "\ufffd" in str(label)
    ]

    if suspicious_labels:
        print(f"\n{name}:")
        print(suspicious_labels)

print("\nLabel normalization check completed.")


Label normalization check completed.


In [17]:
web_dataset = working_datasets[
    "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv"
]

print(
    web_dataset["Label"]
    .value_counts()
    .loc[lambda x: x.index.str.contains("Web Attack")]
)

Label
Web Attack - Brute Force      1507
Web Attack - XSS               652
Web Attack - SQL Injection      21
Name: count, dtype: int64


In [18]:
reference_columns = list(
    next(iter(working_datasets.values())).columns
)

for name, df in working_datasets.items():

    same_columns = set(df.columns) == set(reference_columns)

    same_order = list(df.columns) == reference_columns

    print(f"\nDataset: {name}")
    print("Same columns:", same_columns)
    print("Same column order:", same_order)
    print("Number of columns:", len(df.columns))


Dataset: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Friday-WorkingHours-Morning.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Monday-WorkingHours.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Tuesday-WorkingHours.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79

Dataset: Wednesday-workingHours.pcap_ISCX.csv
Same columns: True
Same column order: True
Number of columns: 79


In [19]:
for name, df in working_datasets.items():

    print(
        f"{name} | Rows: {df.shape[0]} | Columns: {df.shape[1]}"
    )

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv | Rows: 225745 | Columns: 79
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv | Rows: 286467 | Columns: 79
Friday-WorkingHours-Morning.pcap_ISCX.csv | Rows: 191033 | Columns: 79
Monday-WorkingHours.pcap_ISCX.csv | Rows: 529918 | Columns: 79
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv | Rows: 288602 | Columns: 79
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv | Rows: 170366 | Columns: 79
Tuesday-WorkingHours.pcap_ISCX.csv | Rows: 445909 | Columns: 79
Wednesday-workingHours.pcap_ISCX.csv | Rows: 692703 | Columns: 79


In [20]:
all_data = pd.concat(
    working_datasets.values(),
    ignore_index=True
)

print("Merged dataset shape:", all_data.shape)

Merged dataset shape: (2830743, 79)


In [21]:
expected_rows = sum(
    len(df) for df in working_datasets.values()
)

actual_rows = len(all_data)

print("Expected rows:", expected_rows)
print("Actual rows:", actual_rows)

print("Rows match:", expected_rows == actual_rows)

Expected rows: 2830743
Actual rows: 2830743
Rows match: True


In [22]:
print("Shape:", all_data.shape)

print("\nData types:")
print(all_data.dtypes.value_counts())

print("\nFirst 5 rows:")
display(all_data.head())

Shape: (2830743, 79)

Data types:
int64      54
float64    24
str         1
Name: count, dtype: int64

First 5 rows:


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [23]:
source_datasets = {}

for name, df in working_datasets.items():

    temp_df = df.copy()

    temp_df["Source_Dataset"] = name

    source_datasets[name] = temp_df


all_data = pd.concat(
    source_datasets.values(),
    ignore_index=True
)

In [24]:
print("Final shape:", all_data.shape)

print("\nSource Dataset distribution:")
display(all_data["Source_Dataset"].value_counts())

print("\nMissing Source values:",
      all_data["Source_Dataset"].isna().sum())

Final shape: (2830743, 80)

Source Dataset distribution:


Source_Dataset
Wednesday-workingHours.pcap_ISCX.csv                           692703
Monday-WorkingHours.pcap_ISCX.csv                              529918
Tuesday-WorkingHours.pcap_ISCX.csv                             445909
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv    288602
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv           286467
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv               225745
Friday-WorkingHours-Morning.pcap_ISCX.csv                      191033
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv         170366
Name: count, dtype: int64


Missing Source values: 0


In [25]:
# Count duplicated rows across all columns
exact_duplicates = all_data.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

print(
    "Duplicate percentage:",
    round(exact_duplicates / len(all_data) * 100, 4),
    "%"
)

Exact duplicate rows: 256479
Duplicate percentage: 9.0605 %


In [26]:
duplicate_report = []

for name, df in all_data.groupby("Source_Dataset"):

    duplicates = df.duplicated().sum()

    duplicate_report.append({
        "Dataset": name,
        "Total Rows": len(df),
        "Duplicate Rows": duplicates,
        "Duplicate Percentage": round(
            duplicates / len(df) * 100, 4
        )
    })

duplicate_report = pd.DataFrame(duplicate_report)

display(duplicate_report)

,Dataset,Total Rows,Duplicate Rows,Duplicate Percentage
0,Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv,225745,2633,1.1664
1,Friday-WorkingHours-Afternoon-PortScan.pcap_IS...,286467,72353,25.2570
2,Friday-WorkingHours-Morning.pcap_ISCX.csv,191033,6888,3.6057
3,Monday-WorkingHours.pcap_ISCX.csv,529918,26935,5.0829
4,Thursday-WorkingHours-Afternoon-Infilteration....,288602,35630,12.3457
5,Thursday-WorkingHours-Morning-WebAttacks.pcap_...,170366,6066,3.5606
6,Tuesday-WorkingHours.pcap_ISCX.csv,445909,24065,5.3968
7,Wednesday-workingHours.pcap_ISCX.csv,692703,81909,11.8245


In [27]:
# Select network traffic features only
feature_cols = [
    col for col in all_data.columns
    if col not in ["Label", "Source_Dataset"]
]

print("Number of features:", len(feature_cols))

Number of features: 78


In [28]:
# Generate a hash for each feature row
feature_hashes = pd.util.hash_pandas_object(
    all_data[feature_cols],
    index=False
)

hash_label_df = pd.DataFrame({
    "Feature_Hash": feature_hashes,
    "Label": all_data["Label"]
})

# Count unique labels for each feature pattern
labels_per_pattern = (
    hash_label_df
    .groupby("Feature_Hash")["Label"]
    .nunique()
)

# Patterns associated with more than one label
conflicting_hashes = labels_per_pattern[
    labels_per_pattern > 1
]

print(
    "Number of conflicting feature patterns:",
    len(conflicting_hashes)
)

Number of conflicting feature patterns: 698


In [29]:
conflicting_mask = feature_hashes.isin(
    conflicting_hashes.index
)

conflicting_rows = all_data[conflicting_mask]

print(
    "Total rows involved in label conflicts:",
    len(conflicting_rows)
)

print("\nLabels involved:")
display(conflicting_rows["Label"].value_counts())

print("\nSource datasets:")
display(conflicting_rows["Source_Dataset"].value_counts())

Total rows involved in label conflicts: 7020

Labels involved:


Label
DoS Hulk         5224
PortScan         1053
BENIGN            736
DDoS                6
DoS slowloris       1
Name: count, dtype: int64


Source datasets:


Source_Dataset
Wednesday-workingHours.pcap_ISCX.csv                           5274
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv           1060
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv     620
Tuesday-WorkingHours.pcap_ISCX.csv                               33
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                 11
Friday-WorkingHours-Morning.pcap_ISCX.csv                        10
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv            9
Monday-WorkingHours.pcap_ISCX.csv                                 3
Name: count, dtype: int64

In [30]:
# Create a temporary verification DataFrame
conflict_check = all_data.loc[
    conflicting_mask,
    feature_cols + ["Label", "Source_Dataset"]
].copy()

conflict_check["Feature_Hash"] = (
    feature_hashes.loc[conflicting_mask].values
)

verification_results = []

# Verify every conflicting hash group
for hash_value, group in conflict_check.groupby("Feature_Hash"):

    unique_features = group[feature_cols].drop_duplicates()

    verification_results.append({
        "Feature_Hash": hash_value,
        "Number of Rows": len(group),
        "Number of Labels": group["Label"].nunique(),
        "Unique Feature Rows": len(unique_features),
        "Labels": group["Label"].unique().tolist()
    })

verification_df = pd.DataFrame(verification_results)

# Summarize verification
exact_conflicts = verification_df[
    verification_df["Unique Feature Rows"] == 1
]

non_exact_groups = verification_df[
    verification_df["Unique Feature Rows"] > 1
]

print("Total conflicting hash groups:", len(verification_df))

print("Confirmed exact conflicting groups:", len(exact_conflicts))

print("Groups requiring further investigation:", len(non_exact_groups))

print("\nVerification summary:")
display(
    verification_df["Unique Feature Rows"].value_counts()
)

Total conflicting hash groups: 698
Confirmed exact conflicting groups: 698
Groups requiring further investigation: 0

Verification summary:


Unique Feature Rows
1    698
Name: count, dtype: int64

In [31]:
pattern_summary = (
    conflict_check
    .groupby("Feature_Hash")
    .agg(
        Number_Rows=("Label", "size"),
        Number_Labels=("Label", "nunique"),
        Number_Sources=("Source_Dataset", "nunique"),
        Labels=("Label", lambda x: sorted(x.unique())),
        Sources=("Source_Dataset", lambda x: sorted(x.unique()))
    )
    .reset_index()
)

print("Patterns by number of different labels:")
display(pattern_summary["Number_Labels"].value_counts().sort_index())

print("\nPatterns by number of source datasets:")
display(pattern_summary["Number_Sources"].value_counts().sort_index())

print("\nPatterns found in multiple datasets:")
display(
    pattern_summary[
        pattern_summary["Number_Sources"] > 1
    ].sort_values("Number_Rows", ascending=False).head(10)
)

print("\nPatterns found in one dataset only:")
display(
    pattern_summary[
        pattern_summary["Number_Sources"] == 1
    ].sort_values("Number_Rows", ascending=False).head(10)
)

Patterns by number of different labels:


Number_Labels
2    698
Name: count, dtype: int64


Patterns by number of source datasets:


Number_Sources
1     34
2    658
3      6
Name: count, dtype: int64


Patterns found in multiple datasets:


,Feature_Hash,Number_Rows,Number_Labels,Number_Sources,Labels,Sources
53,1676153123259510862,3362,2,3,"[BENIGN, DoS Hulk]",[Friday-WorkingHours-Afternoon-PortScan.pcap_I...
537,14084560481999829949,354,2,3,"[BENIGN, DoS Hulk]",[Friday-WorkingHours-Afternoon-DDos.pcap_ISCX....
570,14980223192479222106,164,2,2,"[BENIGN, DoS Hulk]","[Friday-WorkingHours-Morning.pcap_ISCX.csv, We..."
185,4855009796914144858,133,2,2,"[BENIGN, DoS Hulk]",[Thursday-WorkingHours-Afternoon-Infilteration...
302,8016126238371002509,54,2,2,"[BENIGN, DoS Hulk]","[Tuesday-WorkingHours.pcap_ISCX.csv, Wednesday..."
437,11557951919960783205,48,2,2,"[BENIGN, DoS Hulk]",[Thursday-WorkingHours-Afternoon-Infilteration...
128,3566029670018468095,47,2,2,"[BENIGN, DoS Hulk]","[Tuesday-WorkingHours.pcap_ISCX.csv, Wednesday..."
439,11568677049683023315,46,2,2,"[BENIGN, DoS Hulk]","[Tuesday-WorkingHours.pcap_ISCX.csv, Wednesday..."
490,12892175988359113389,34,2,2,"[BENIGN, DoS Hulk]","[Tuesday-WorkingHours.pcap_ISCX.csv, Wednesday..."
207,5522707036101724640,32,2,2,"[BENIGN, DoS Hulk]",[Thursday-WorkingHours-Morning-WebAttacks.pcap...



Patterns found in one dataset only:


,Feature_Hash,Number_Rows,Number_Labels,Number_Sources,Labels,Sources
605,16007925024960830336,81,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
341,9025813330109369079,61,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
452,11762083746210125647,36,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
214,5706618276573943364,28,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
164,4368527496468821709,24,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
492,12926834750395675117,21,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
250,6394189048257247861,21,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
236,6108503209309145731,20,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
399,10437217239449385853,16,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]
530,13837314209003905402,16,2,1,"[BENIGN, DoS Hulk]",[Wednesday-workingHours.pcap_ISCX.csv]


In [32]:
# Find the largest conflicting pattern
group_sizes = conflict_check.groupby("Feature_Hash").size()

largest_hash = group_sizes.idxmax()

largest_group = conflict_check[
    conflict_check["Feature_Hash"] == largest_hash
].copy()

print("Largest group rows:", len(largest_group))

print("\nLabels distribution:")
display(largest_group["Label"].value_counts())

print("\nSource distribution:")
display(largest_group["Source_Dataset"].value_counts())

print("\nLabels by source:")
display(
    pd.crosstab(
        largest_group["Source_Dataset"],
        largest_group["Label"]
    )
)

Largest group rows: 3362

Labels distribution:


Label
DoS Hulk    3360
BENIGN         2
Name: count, dtype: int64


Source distribution:


Source_Dataset
Wednesday-workingHours.pcap_ISCX.csv                    3360
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv       1
Tuesday-WorkingHours.pcap_ISCX.csv                         1
Name: count, dtype: int64


Labels by source:


Label,BENIGN,DoS Hulk
Source_Dataset,,
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv,1,0
Tuesday-WorkingHours.pcap_ISCX.csv,1,0
Wednesday-workingHours.pcap_ISCX.csv,0,3360


In [33]:
# Total rows per label
total_per_label = all_data["Label"].value_counts()

# Conflicting rows per label
conflicting_per_label = conflicting_rows["Label"].value_counts()

# Build a comparison table
label_conflict_summary = pd.DataFrame({
    "Total_Rows": total_per_label,
    "Conflicting_Rows": conflicting_per_label
}).fillna(0)

label_conflict_summary["Conflicting_Rows"] = (
    label_conflict_summary["Conflicting_Rows"].astype(int)
)

label_conflict_summary["Conflict_Percentage"] = (
    label_conflict_summary["Conflicting_Rows"]
    / label_conflict_summary["Total_Rows"]
    * 100
)

display(
    label_conflict_summary.sort_values(
        "Conflict_Percentage",
        ascending=False
    )
)

,Total_Rows,Conflicting_Rows,Conflict_Percentage
Label,,,
DoS Hulk,231073,5224,2.260757
PortScan,158930,1053,0.662556
BENIGN,2273097,736,0.032379
DoS slowloris,5796,1,0.017253
DDoS,128027,6,0.004687
DoS GoldenEye,10293,0,0.000000
Bot,1966,0,0.000000
DoS Slowhttptest,5499,0,0.000000
FTP-Patator,7938,0,0.000000


In [34]:
# Count rows for each feature pattern and source
pattern_source_counts = (
    all_data.assign(Feature_Hash=feature_hashes.values)
    .groupby(["Feature_Hash", "Source_Dataset"])
    .size()
    .reset_index(name="Rows")
)

# Count how many feature patterns appear in multiple datasets
sources_per_pattern = (
    pattern_source_counts
    .groupby("Feature_Hash")["Source_Dataset"]
    .nunique()
)

multi_source_patterns = sources_per_pattern[
    sources_per_pattern > 1
]

print(
    "Feature patterns appearing in multiple datasets:",
    len(multi_source_patterns)
)

print(
    "Total rows belonging to these patterns:",
    all_data.assign(Feature_Hash=feature_hashes.values)
    .loc[
        all_data.assign(Feature_Hash=feature_hashes.values)
        ["Feature_Hash"].isin(multi_source_patterns.index)
    ].shape[0]
)

print(
    "\nDistribution by number of source datasets:"
)

display(
    sources_per_pattern.value_counts().sort_index()
)

Feature patterns appearing in multiple datasets: 34208
Total rows belonging to these patterns: 146363

Distribution by number of source datasets:


Source_Dataset
1    2487456
2      23339
3       6262
4       2680
5       1173
6        571
7        179
8          4
Name: count, dtype: int64

In [35]:
# Analyze labels and sources for each feature pattern

pattern_analysis = (
    all_data.assign(Feature_Hash=feature_hashes.values)
    .groupby("Feature_Hash")
    .agg(
        Number_Rows=("Label", "size"),
        Number_Labels=("Label", "nunique"),
        Number_Sources=("Source_Dataset", "nunique")
    )
)

# Patterns appearing in multiple datasets
multi_source_analysis = pattern_analysis[
    pattern_analysis["Number_Sources"] > 1
].copy()

# Split into two categories
same_label_patterns = multi_source_analysis[
    multi_source_analysis["Number_Labels"] == 1
]

conflicting_label_patterns = multi_source_analysis[
    multi_source_analysis["Number_Labels"] > 1
]

print(
    "Multi-source patterns with same label:",
    len(same_label_patterns)
)

print(
    "Multi-source patterns with conflicting labels:",
    len(conflicting_label_patterns)
)

print(
    "\nRows in same-label patterns:",
    same_label_patterns["Number_Rows"].sum()
)

print(
    "Rows in conflicting-label patterns:",
    conflicting_label_patterns["Number_Rows"].sum()
)

Multi-source patterns with same label: 33544
Multi-source patterns with conflicting labels: 664

Rows in same-label patterns: 139797
Rows in conflicting-label patterns: 6566


In [36]:
# 1. Missing values
missing_values = all_data[feature_cols].isna().sum()

print("Total missing values:")
print(missing_values[missing_values > 0])

# 2. Infinite values
infinity_values = pd.Series({
    col: np.isinf(all_data[col]).sum()
    for col in feature_cols
    if pd.api.types.is_numeric_dtype(all_data[col])
})

print("\nInfinite values:")
print(infinity_values[infinity_values > 0])

# 3. Negative values
negative_values = pd.Series({
    col: (all_data[col] < 0).sum()
    for col in feature_cols
    if pd.api.types.is_numeric_dtype(all_data[col])
})

print("\nNegative values:")
print(negative_values[negative_values > 0])

Total missing values:
Flow Bytes/s    1358
dtype: int64

Infinite values:
Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64

Negative values:
Flow Duration                  115
Flow Bytes/s                    85
Flow Packets/s                 115
Flow IAT Mean                  115
Flow IAT Max                   115
Flow IAT Min                  2891
Fwd IAT Min                     17
Fwd Header Length               35
Bwd Header Length               22
Fwd Header Length.1             35
Init_Win_bytes_forward     1001189
Init_Win_bytes_backward    1441552
min_seg_size_forward            35
dtype: int64


In [37]:
negative_cols = negative_values[negative_values > 0].index

for col in negative_cols:

    negative_data = all_data.loc[
        all_data[col] < 0, col
    ]

    print("\n" + "=" * 60)
    print("Column:", col)
    print("Negative rows:", len(negative_data))
    print("Unique negative values:", negative_data.nunique())

    print("\nMost common negative values:")
    display(negative_data.value_counts().head(10))


Column: Flow Duration
Negative rows: 115
Unique negative values: 5

Most common negative values:


Flow Duration
-1     109
-12      2
-2       2
-13      1
-4       1
Name: count, dtype: int64


Column: Flow Bytes/s
Negative rows: 85
Unique negative values: 9

Most common negative values:


Flow Bytes/s
-1.200000e+07    50
-6.000000e+06    16
-8.000000e+06    13
-1.930000e+08     1
-6.666667e+05     1
-4.615385e+05     1
-4.000000e+06     1
-1.000000e+06     1
-2.610000e+08     1
Name: count, dtype: int64


Column: Flow Packets/s
Negative rows: 115
Unique negative values: 5

Most common negative values:


Flow Packets/s
-2.000000e+06    109
-1.666667e+05      2
-1.000000e+06      2
-1.538462e+05      1
-5.000000e+05      1
Name: count, dtype: int64

... (16 outputs left out)


Column: Init_Win_bytes_backward
Negative rows: 1441552
Unique negative values: 1

Most common negative values:


Init_Win_bytes_backward
-1    1441552
Name: count, dtype: int64


Column: min_seg_size_forward
Negative rows: 35
Unique negative values: 4

Most common negative values:


min_seg_size_forward
-83885313     23
-536870660     9
-536870661     2
-1             1
Name: count, dtype: int64

In [38]:
# Columns with suspicious negative values
suspicious_cols = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Header Length.1",
    "min_seg_size_forward"
]

for col in suspicious_cols:

    negative_rows = all_data[all_data[col] < 0]

    print("\n" + "=" * 70)
    print("Column:", col)
    print("Negative rows:", len(negative_rows))

    print("\nDistribution by Label:")
    display(negative_rows["Label"].value_counts())

    print("\nDistribution by Dataset:")
    display(negative_rows["Source_Dataset"].value_counts())


Column: Flow Duration
Negative rows: 115

Distribution by Label:


Label
BENIGN    115
Name: count, dtype: int64


Distribution by Dataset:


Source_Dataset
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv           36
Wednesday-workingHours.pcap_ISCX.csv                           21
Tuesday-WorkingHours.pcap_ISCX.csv                             17
Monday-WorkingHours.pcap_ISCX.csv                              15
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv         11
Friday-WorkingHours-Morning.pcap_ISCX.csv                       9
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv     4
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv                2
Name: count, dtype: int64


Column: Flow Bytes/s
Negative rows: 85

Distribution by Label:


Label
BENIGN    85
Name: count, dtype: int64

... (34 outputs left out)


Column: min_seg_size_forward
Negative rows: 35

Distribution by Label:


Label
BENIGN    35
Name: count, dtype: int64


Distribution by Dataset:


Source_Dataset
Monday-WorkingHours.pcap_ISCX.csv       23
Tuesday-WorkingHours.pcap_ISCX.csv      11
Wednesday-workingHours.pcap_ISCX.csv     1
Name: count, dtype: int64

In [39]:
# Create masks for suspicious negative values

negative_masks = {
    "Flow Duration": all_data["Flow Duration"] < 0,
    "Flow Bytes/s": all_data["Flow Bytes/s"] < 0,
    "Flow Packets/s": all_data["Flow Packets/s"] < 0,
    "Flow IAT Mean": all_data["Flow IAT Mean"] < 0,
    "Flow IAT Max": all_data["Flow IAT Max"] < 0,
    "Flow IAT Min": all_data["Flow IAT Min"] < 0,
    "Fwd IAT Min": all_data["Fwd IAT Min"] < 0,
    "Fwd Header Length": all_data["Fwd Header Length"] < 0,
    "Bwd Header Length": all_data["Bwd Header Length"] < 0,
    "min_seg_size_forward": all_data["min_seg_size_forward"] < 0
}

# Count how many rows are affected by each issue
negative_summary = pd.DataFrame({
    "Negative_Rows": {
        col: mask.sum()
        for col, mask in negative_masks.items()
    }
})

display(negative_summary)

# Check overlap between the main suspicious groups
overlap_summary = pd.DataFrame({
    "Flow Duration": negative_masks["Flow Duration"],
    "Flow Bytes/s": negative_masks["Flow Bytes/s"],
    "Flow IAT Min": negative_masks["Flow IAT Min"],
    "Fwd Header Length": negative_masks["Fwd Header Length"],
    "Bwd Header Length": negative_masks["Bwd Header Length"],
    "min_seg_size_forward": negative_masks["min_seg_size_forward"]
})

print("Rows affected by at least one suspicious negative value:")
print(overlap_summary.any(axis=1).sum())

print("\nRows affected by multiple suspicious columns:")
print((overlap_summary.sum(axis=1) > 1).sum())

print("\nOverlap between the 115 negative Flow Duration rows:")
display(
    overlap_summary.loc[
        negative_masks["Flow Duration"]
    ].sum().sort_values(ascending=False)
)

,Negative_Rows
Flow Duration,115
Flow Bytes/s,85
Flow Packets/s,115
Flow IAT Mean,115
Flow IAT Max,115
Flow IAT Min,2891
Fwd IAT Min,17
Fwd Header Length,35
Bwd Header Length,22
min_seg_size_forward,35


Rows affected by at least one suspicious negative value:
2926

Rows affected by multiple suspicious columns:
150

Overlap between the 115 negative Flow Duration rows:


Flow Duration           115
Flow IAT Min            115
Flow Bytes/s             85
Fwd Header Length         0
Bwd Header Length         0
min_seg_size_forward      0
dtype: int64

In [40]:
# Include all suspicious negative columns
all_negative_cols = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Header Length.1",
    "min_seg_size_forward"
]

# Build a complete negative mask
all_negative_masks = pd.DataFrame({
    col: all_data[col] < 0
    for col in all_negative_cols
})

# Number of negative columns per row
negative_columns_per_row = all_negative_masks.sum(axis=1)

print("Rows affected by at least one negative value:")
print((negative_columns_per_row > 0).sum())

print("\nRows affected by multiple negative columns:")
print((negative_columns_per_row > 1).sum())

print("\nDistribution by number of negative columns:")
display(negative_columns_per_row.value_counts().sort_index())

# Check specific overlaps
print("\nFwd Header Length and min_seg_size_forward overlap:")
print(
    (
        all_negative_masks["Fwd Header Length"]
        & all_negative_masks["min_seg_size_forward"]
    ).sum()
)

print("\nFwd Header Length and Bwd Header Length overlap:")
print(
    (
        all_negative_masks["Fwd Header Length"]
        & all_negative_masks["Bwd Header Length"]
    ).sum()
)

print("\nFwd IAT Min and Flow IAT Min overlap:")
print(
    (
        all_negative_masks["Fwd IAT Min"]
        & all_negative_masks["Flow IAT Min"]
    ).sum()
)

Rows affected by at least one negative value:
2926

Rows affected by multiple negative columns:
167

Distribution by number of negative columns:


0    2827817
1       2759
2         17
3         13
4         22
5         30
6         85
Name: count, dtype: int64


Fwd Header Length and min_seg_size_forward overlap:
35

Fwd Header Length and Bwd Header Length overlap:
22

Fwd IAT Min and Flow IAT Min overlap:
17


In [41]:
invalid_negative_cols = [
    "Flow Duration",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Max",
    "Flow IAT Min",
    "Fwd IAT Min",
    "Fwd Header Length",
    "Bwd Header Length",
    "Fwd Header Length.1",
    "min_seg_size_forward"
]

# Create a separate cleaned working copy
clean_data = all_data.copy()

# Convert invalid negative measurements to NaN
for col in invalid_negative_cols:

    negative_mask = clean_data[col] < 0

    negative_count = negative_mask.sum()

    clean_data.loc[negative_mask, col] = np.nan

    print(f"{col}: {negative_count} values converted to NaN")

print("\nCleaning completed.")

Flow Duration: 115 values converted to NaN
Flow Bytes/s: 85 values converted to NaN
Flow Packets/s: 115 values converted to NaN
Flow IAT Mean: 115 values converted to NaN
Flow IAT Max: 115 values converted to NaN
Flow IAT Min: 2891 values converted to NaN
Fwd IAT Min: 17 values converted to NaN
Fwd Header Length: 35 values converted to NaN
Bwd Header Length: 22 values converted to NaN
Fwd Header Length.1: 35 values converted to NaN
min_seg_size_forward: 35 values converted to NaN

Cleaning completed.


In [42]:
remaining_negative_values = pd.Series({
    col: (clean_data[col] < 0).sum()
    for col in invalid_negative_cols
})

print("Remaining negative values:")
display(remaining_negative_values)

print("\nMissing values after cleaning:")

missing_after_cleaning = clean_data[feature_cols].isna().sum()

display(
    missing_after_cleaning[missing_after_cleaning > 0]
)

print("\nDataset shape:")
print(clean_data.shape)

Remaining negative values:


Flow Duration           0
Flow Bytes/s            0
Flow Packets/s          0
Flow IAT Mean           0
Flow IAT Max            0
Flow IAT Min            0
Fwd IAT Min             0
Fwd Header Length       0
Bwd Header Length       0
Fwd Header Length.1     0
min_seg_size_forward    0
dtype: int64


Missing values after cleaning:


Flow Duration            115
Flow Bytes/s            1443
Flow Packets/s           115
Flow IAT Mean            115
Flow IAT Max             115
Flow IAT Min            2891
Fwd IAT Min               17
Fwd Header Length         35
Bwd Header Length         22
Fwd Header Length.1       35
min_seg_size_forward      35
dtype: int64


Dataset shape:
(2830743, 80)


In [43]:
# Count infinity values before cleaning
infinity_before = pd.Series({
    col: np.isinf(clean_data[col]).sum()
    for col in feature_cols
})

print("Infinity values before cleaning:")
display(infinity_before[infinity_before > 0])

# Replace positive and negative infinity with NaN
clean_data[feature_cols] = clean_data[feature_cols].replace(
    [np.inf, -np.inf],
    np.nan
)

print("\nInfinity values converted to NaN.")

Infinity values before cleaning:


Flow Bytes/s      1509
Flow Packets/s    2867
dtype: int64


Infinity values converted to NaN.


In [44]:
infinity_after = pd.Series({
    col: np.isinf(clean_data[col]).sum()
    for col in feature_cols
})

print("Remaining Infinity values:")
display(infinity_after[infinity_after > 0])

print("\nTotal missing values now:")

missing_total = clean_data[feature_cols].isna().sum().sum()

print(missing_total)

print("\nDataset shape:")
print(clean_data.shape)

Remaining Infinity values:


Series([], dtype: int64)


Total missing values now:
9314

Dataset shape:
(2830743, 80)


In [45]:
missing_summary = pd.DataFrame({
    "Missing_Count": clean_data[feature_cols].isna().sum(),
    "Missing_Percentage": (
        clean_data[feature_cols].isna().mean() * 100
    )
})

missing_summary = missing_summary[
    missing_summary["Missing_Count"] > 0
].sort_values(
    by="Missing_Count",
    ascending=False
)

display(missing_summary)

,Missing_Count,Missing_Percentage
Flow Packets/s,2982,0.105343
Flow Bytes/s,2952,0.104284
Flow IAT Min,2891,0.102129
Flow Duration,115,0.004063
Flow IAT Mean,115,0.004063
Flow IAT Max,115,0.004063
Fwd Header Length,35,0.001236
min_seg_size_forward,35,0.001236
Fwd Header Length.1,35,0.001236
Bwd Header Length,22,0.000777


In [46]:
print("Total missing cells:",
      clean_data[feature_cols].isna().sum().sum())

print("Total missing percentage:",
      clean_data[feature_cols].isna().sum().sum()
      / (len(clean_data) * len(feature_cols)) * 100)

Total missing cells: 9314
Total missing percentage: 0.004218336189836252


In [47]:
print("Missing Labels:")
print(clean_data["Label"].isna().sum())

print("\nNumber of Classes:")
print(clean_data["Label"].nunique())

print("\nLabel Distribution:")
display(
    clean_data["Label"].value_counts().to_frame("Count")
)

print("\nTotal Rows:")
print(len(clean_data))

Missing Labels:
0

Number of Classes:
15

Label Distribution:


,Count
Label,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499



Total Rows:
2830743


In [48]:
# Exclude target and metadata columns
feature_nunique = clean_data[feature_cols].nunique(dropna=False)

# Get constant features
constant_features = feature_nunique[
    feature_nunique == 1
].index.tolist()

print("Number of constant features:", len(constant_features))

print("\nConstant Features:")

constant_summary = pd.DataFrame({
    "Feature": constant_features,
    "Unique_Values": [
        clean_data[col].iloc[0]
        for col in constant_features
    ]
})

display(constant_summary)

Number of constant features: 8

Constant Features:


,Feature,Unique_Values
0,Bwd PSH Flags,0
1,Bwd URG Flags,0
2,Fwd Avg Bytes/Bulk,0
3,Fwd Avg Packets/Bulk,0
4,Fwd Avg Bulk Rate,0
5,Bwd Avg Bytes/Bulk,0
6,Bwd Avg Packets/Bulk,0
7,Bwd Avg Bulk Rate,0


In [49]:
print("Shape before removing constant features:")
print(clean_data.shape)

clean_data = clean_data.drop(columns=constant_features)

print("\nShape after removing constant features:")
print(clean_data.shape)

print("\nRemaining constant features:")

remaining_constants = [
    col for col in clean_data.columns
    if col not in ["Label", "Source_Dataset"]
    and clean_data[col].nunique(dropna=False) == 1
]

print(remaining_constants)

Shape before removing constant features:
(2830743, 80)

Shape after removing constant features:
(2830743, 72)

Remaining constant features:
[]


In [50]:
duplicate_feature_pairs = [
    ("Fwd Header Length", "Fwd Header Length.1"),
    ("Fwd PSH Flags", "SYN Flag Count"),
    ("Total Backward Packets", "Subflow Bwd Packets"),
    ("Total Fwd Packets", "Subflow Fwd Packets")
]

for col1, col2 in duplicate_feature_pairs:

    if col1 in clean_data.columns and col2 in clean_data.columns:

        identical = clean_data[col1].equals(clean_data[col2])

        print(f"{col1} == {col2}: {identical}")

    else:
        print(f"Missing column: {col1} or {col2}")

Fwd Header Length == Fwd Header Length.1: True
Fwd PSH Flags == SYN Flag Count: True
Total Backward Packets == Subflow Bwd Packets: True
Total Fwd Packets == Subflow Fwd Packets: True


In [51]:
duplicate_features_to_drop = [
    "Fwd Header Length.1",
    "SYN Flag Count",
    "Subflow Bwd Packets",
    "Subflow Fwd Packets"
]

print("Shape before removing duplicate features:")
print(clean_data.shape)

clean_data = clean_data.drop(
    columns=duplicate_features_to_drop
)

print("\nShape after removing duplicate features:")
print(clean_data.shape)

print("\nRemoved features:")
print(duplicate_features_to_drop)

Shape before removing duplicate features:
(2830743, 72)

Shape after removing duplicate features:
(2830743, 68)

Removed features:
['Fwd Header Length.1', 'SYN Flag Count', 'Subflow Bwd Packets', 'Subflow Fwd Packets']


In [52]:
# Select numerical features only
numerical_features = clean_data.select_dtypes(
    include=[np.number]
).columns.tolist()

print("Number of numerical features:", len(numerical_features))

# Calculate correlation matrix
correlation_matrix = clean_data[
    numerical_features
].corr(method="pearson")

Number of numerical features: 66


In [53]:
# Get upper triangle to avoid duplicate pairs
upper_triangle = correlation_matrix.where(
    np.triu(
        np.ones(correlation_matrix.shape),
        k=1
    ).astype(bool)
)

high_correlation_pairs = (
    upper_triangle.stack()
    .reset_index()
)

high_correlation_pairs.columns = [
    "Feature_1",
    "Feature_2",
    "Correlation"
]

high_correlation_pairs["Abs_Correlation"] = (
    high_correlation_pairs["Correlation"].abs()
)

high_correlation_pairs = high_correlation_pairs[
    high_correlation_pairs["Abs_Correlation"] >= 0.95
].sort_values(
    by="Abs_Correlation",
    ascending=False
)

print("Number of highly correlated pairs:",
      len(high_correlation_pairs))

display(high_correlation_pairs)

Number of highly correlated pairs: 37


,Feature_1,Feature_2,Correlation,Abs_Correlation
843,Bwd Packet Length Mean,Avg Bwd Segment Size,1.000000,1.000000
578,Fwd Packet Length Mean,Avg Fwd Segment Size,1.000000,1.000000
2092,Fwd URG Flags,CWE Flag Count,1.000000,1.000000
383,Total Length of Bwd Packets,Subflow Bwd Bytes,1.000000,1.000000
316,Total Length of Fwd Packets,Subflow Fwd Bytes,0.999999,0.999999
231,Total Backward Packets,Bwd Header Length,0.999562,0.999562
164,Total Fwd Packets,Fwd Header Length,0.999490,0.999490
135,Total Fwd Packets,Total Backward Packets,0.999070,0.999070
2145,Fwd Header Length,Bwd Header Length,0.998808,0.998808
230,Total Backward Packets,Fwd Header Length,0.998778,0.998778


In [54]:
exact_duplicate_pairs = []

for _, row in high_correlation_pairs.iterrows():

    col1 = row["Feature_1"]
    col2 = row["Feature_2"]

    # Check equality, including matching NaN positions
    identical = (
        clean_data[col1].eq(clean_data[col2])
        | (
            clean_data[col1].isna()
            & clean_data[col2].isna()
        )
    ).all()

    if identical:
        exact_duplicate_pairs.append((col1, col2))

print("Number of exact duplicate pairs:",
      len(exact_duplicate_pairs))

print("\nExact duplicate pairs:")

for pair in exact_duplicate_pairs:
    print(pair)

Number of exact duplicate pairs: 1

Exact duplicate pairs:
('Fwd URG Flags', 'CWE Flag Count')


In [55]:
duplicate_features_to_drop = [
    "CWE Flag Count"
]

print("Shape before removal:")
print(clean_data.shape)

clean_data = clean_data.drop(
    columns=duplicate_features_to_drop
)

print("\nShape after removal:")
print(clean_data.shape)

print("\nRemaining columns:")
print(clean_data.shape[1])

Shape before removal:
(2830743, 68)

Shape after removal:
(2830743, 67)

Remaining columns:
67


In [56]:
print("Data Types Distribution:")
display(clean_data.dtypes.value_counts())

print("\nNon-Numerical Columns:")

non_numerical_cols = clean_data.select_dtypes(
    exclude=[np.number]
).columns.tolist()

print(non_numerical_cols)

print("\nNumerical Features:")
numerical_features = clean_data.select_dtypes(
    include=[np.number]
).columns.tolist()

print(len(numerical_features))

Data Types Distribution:


int64      34
float64    31
str         2
Name: count, dtype: int64


Non-Numerical Columns:
['Label', 'Source_Dataset']

Numerical Features:
65


In [57]:
feature_scale_summary = clean_data[numerical_features].describe().T

feature_scale_summary = feature_scale_summary[
    ["min", "25%", "50%", "75%", "max", "mean", "std"]
]

display(feature_scale_summary)

,min,25%,50%,75%,max,mean,std
Destination Port,0.0,53.0,80.0,443.0,65535.0,8.071483e+03,1.828363e+04
Flow Duration,0.0,155.0,31317.0,3206060.0,119999998.0,1.478626e+07,3.365430e+07
Total Fwd Packets,1.0,2.0,2.0,5.0,219759.0,9.361160e+00,7.496728e+02
Total Backward Packets,0.0,1.0,2.0,4.0,291922.0,1.039377e+01,9.973883e+02
Total Length of Fwd Packets,0.0,12.0,62.0,187.0,12900000.0,5.493024e+02,9.993589e+03
...,...,...,...,...,...,...,...
Active Min,0.0,0.0,0.0,0.0,110000000.0,5.829582e+04,5.770923e+05
Idle Mean,0.0,0.0,0.0,0.0,120000000.0,8.316037e+06,2.363008e+07
Idle Std,0.0,0.0,0.0,0.0,76900000.0,5.038439e+05,4.602984e+06
Idle Max,0.0,0.0,0.0,0.0,120000000.0,8.695752e+06,2.436689e+07


In [58]:
skewness = clean_data[numerical_features].skew()

skewness_summary = pd.DataFrame({
    "Skewness": skewness,
    "Abs_Skewness": skewness.abs()
})

skewness_summary = skewness_summary.sort_values(
    by="Abs_Skewness",
    ascending=False
)

display(skewness_summary)

,Skewness,Abs_Skewness
Total Length of Fwd Packets,805.570539,805.570539
Subflow Fwd Bytes,803.598592,803.598592
act_data_pkt_fwd,284.595228,284.595228
Total Backward Packets,244.679507,244.679507
Total Fwd Packets,244.380553,244.380553
...,...,...
Flow Duration,2.155287,2.155287
Destination Port,2.072052,2.072052
PSH Flag Count,0.882925,0.882925
ACK Flag Count,0.792321,0.792321


In [59]:
print("Approximately symmetric (|skew| <= 0.5):",
      (skewness.abs() <= 0.5).sum())

print("Moderately skewed (0.5 < |skew| <= 1):",
      ((skewness.abs() > 0.5) & (skewness.abs() <= 1)).sum())

print("Highly skewed (|skew| > 1):",
      (skewness.abs() > 1).sum())

Approximately symmetric (|skew| <= 0.5): 1
Moderately skewed (0.5 < |skew| <= 1): 2
Highly skewed (|skew| > 1): 62


In [60]:
zero_counts = (
    clean_data[numerical_features]
    .eq(0)
    .sum()
)

zero_summary = pd.DataFrame({
    "Zero_Count": zero_counts,
    "Zero_Percentage": (
        zero_counts / len(clean_data) * 100
    )
})

zero_summary = zero_summary.sort_values(
    by="Zero_Percentage",
    ascending=False
)

display(zero_summary)

,Zero_Count,Zero_Percentage
Fwd URG Flags,2830428,99.988872
RST Flag Count,2830057,99.975766
ECE Flag Count,2830054,99.975660
FIN Flag Count,2730592,96.462024
Fwd PSH Flags,2699265,95.355354
...,...,...
min_seg_size_forward,1827,0.064541
Fwd Header Length,1758,0.062104
Destination Port,1696,0.059914
Total Fwd Packets,0,0.000000


In [61]:
print("Features with more than 50% zeros:",
      (zero_summary["Zero_Percentage"] > 50).sum())

print("Features with more than 90% zeros:",
      (zero_summary["Zero_Percentage"] > 90).sum())

Features with more than 50% zeros: 21
Features with more than 90% zeros: 8


In [62]:
binary_features = []

for col in numerical_features:

    unique_values = set(
        clean_data[col].dropna().unique()
    )

    if unique_values.issubset({0, 1}):
        binary_features.append(col)

print("Number of binary features:", len(binary_features))

print("\nBinary Features:")
print(binary_features)

Number of binary features: 8

Binary Features:
['Fwd PSH Flags', 'Fwd URG Flags', 'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count']


In [63]:
for col in binary_features:

    print(f"\n{col}")

    display(
        clean_data[col].value_counts(dropna=False)
    )


Fwd PSH Flags


Fwd PSH Flags
0    2699265
1     131478
Name: count, dtype: int64


Fwd URG Flags


Fwd URG Flags
0    2830428
1        315
Name: count, dtype: int64


FIN Flag Count


FIN Flag Count
0    2730592
1     100151
Name: count, dtype: int64

... (6 outputs left out)


URG Flag Count


URG Flag Count
0    2562323
1     268420
Name: count, dtype: int64


ECE Flag Count


ECE Flag Count
0    2830054
1        689
Name: count, dtype: int64

In [64]:
negative_summary = pd.DataFrame({
    "Negative_Count": (
        clean_data[numerical_features] < 0
    ).sum()
})

negative_summary = negative_summary[
    negative_summary["Negative_Count"] > 0
].sort_values(
    by="Negative_Count",
    ascending=False
)

display(negative_summary)

,Negative_Count
Init_Win_bytes_backward,1441552
Init_Win_bytes_forward,1001189


In [65]:
sentinel_features = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

for col in sentinel_features:

    print(f"\n{col}")

    display(
        clean_data[col].value_counts(dropna=False).head(15)
    )


Init_Win_bytes_forward


Init_Win_bytes_forward
-1        1001189
 8192      322819
 29200     317513
 1024      142418
 256       101202
 0          99771
 251        96994
 65535      73983
 274        65899
 229        39423
 253        13958
 122        13047
 254        11689
 60         11590
 255        10956
Name: count, dtype: int64


Init_Win_bytes_backward


Init_Win_bytes_backward
-1        1441552
 0         271893
 235       178098
 229       104516
 256        42608
 31         18791
 33304      18319
 60         17744
 29200      16131
 65535      15046
 946        13959
 255        13940
 32832      13534
 122        11388
 939        11242
Name: count, dtype: int64

In [66]:
sentinel_features = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

for col in sentinel_features:

    print("=" * 60)
    print(f"Feature: {col}")

    sentinel_analysis = clean_data.groupby("Label")[col].agg(
        Total_Rows="count",
        Sentinel_Count=lambda x: (x == -1).sum()
    )

    sentinel_analysis["Sentinel_Percentage"] = (
        sentinel_analysis["Sentinel_Count"]
        / sentinel_analysis["Total_Rows"]
    ) * 100

    sentinel_analysis = sentinel_analysis.sort_values(
        by="Sentinel_Percentage",
        ascending=False
    )

    display(sentinel_analysis)
    

Feature: Init_Win_bytes_forward


,Total_Rows,Sentinel_Count,Sentinel_Percentage
Label,,,
BENIGN,2273097,1001182,44.044843
PortScan,158930,7,0.004404
Bot,1966,0,0.000000
DoS GoldenEye,10293,0,0.000000
DoS Hulk,231073,0,0.000000
DoS Slowhttptest,5499,0,0.000000
DDoS,128027,0,0.000000
DoS slowloris,5796,0,0.000000
FTP-Patator,7938,0,0.000000


Feature: Init_Win_bytes_backward


,Total_Rows,Sentinel_Count,Sentinel_Percentage
Label,,,
BENIGN,2273097,1318981,58.025724
DoS Slowhttptest,5499,3172,57.683215
DDoS,128027,46528,36.342334
DoS Hulk,231073,67060,29.021132
DoS slowloris,5796,1642,28.329883
DoS GoldenEye,10293,2579,25.055863
FTP-Patator,7938,1494,18.820862
Infiltration,36,3,8.333333
Web Attack - XSS,652,17,2.607362


In [67]:
sentinel_features = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

for col in sentinel_features:

    print("=" * 80)
    print(f"Feature: {col}")

    source_analysis = clean_data.groupby(
        ["Source_Dataset", "Label"]
    )[col].agg(
        Total_Rows="count",
        Sentinel_Count=lambda x: (x == -1).sum()
    )

    source_analysis["Sentinel_Percentage"] = (
        source_analysis["Sentinel_Count"]
        / source_analysis["Total_Rows"]
    ) * 100

    display(source_analysis)

Feature: Init_Win_bytes_forward


Total_Rows  \
Source_Dataset                                     Label                                    
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   BENIGN                           97718   
                                                   DDoS                            128027   
Friday-WorkingHours-Afternoon-PortScan.pcap_ISC... BENIGN                          127537   
                                                   PortScan                        158930   
Friday-WorkingHours-Morning.pcap_ISCX.csv          BENIGN                          189067   
                                                   Bot                               1966   
Monday-WorkingHours.pcap_ISCX.csv                  BENIGN                          529918   
Thursday-WorkingHours-Afternoon-Infilteration.p... BENIGN                          288566   
                                                   Infiltration                        36   
Thursday-WorkingHours-Morning-WebAttacks.pcap_I... BENIGN                          168186   
                                                   Web Attack - Brute Force          1507   
                                                   Web Attack - SQL Injection          21   
                                                   Web Attack - XSS                   652   
Tuesday-WorkingHours.pcap_ISCX.csv                 BENIGN                          432074   
                                                   FTP-Patator                       7938   
                                                   SSH-Patator                       5897   
Wednesday-workingHours.pcap_ISCX.csv               BENIGN                          440031   
                                                   DoS GoldenEye                    10293   
                                                   DoS Hulk                        231073   
                                                   DoS Slowhttptest                  5499   
                                                   DoS slowloris                     5796   
                                                   Heartbleed                          11   

                                                                               Sentinel_Count  \
Source_Dataset                                     Label                                        
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   BENIGN                               32925   
                                                   DDoS                                     0   
Friday-WorkingHours-Afternoon-PortScan.pcap_ISC... BENIGN                               60203   
                                                   PortScan                                 7   
Friday-WorkingHours-Morning.pcap_ISCX.csv          BENIGN                               95834   
                                                   Bot                                      0   
Monday-WorkingHours.pcap_ISCX.csv                  BENIGN                              224495   
Thursday-WorkingHours-Afternoon-Infilteration.p... BENIGN                              102433   
                                                   Infiltration                             0   
Thursday-WorkingHours-Morning-WebAttacks.pcap_I... BENIGN                               81911   
                                                   Web Attack - Brute Force                 0   
                                                   Web Attack - SQL Injection               0   
                                                   Web Attack - XSS                         0   
Tuesday-WorkingHours.pcap_ISCX.csv                 BENIGN                              200128   
                                                   FTP-Patator                              0   
                                                   SSH-Patator                              0   
Wednesday-workingHours.pcap_ISCX.csv               BENIGN                              203253   
   

Feature: Init_Win_bytes_backward


Total_Rows  \
Source_Dataset                                     Label                                    
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   BENIGN                           97718   
                                                   DDoS                            128027   
Friday-WorkingHours-Afternoon-PortScan.pcap_ISC... BENIGN                          127537   
                                                   PortScan                        158930   
Friday-WorkingHours-Morning.pcap_ISCX.csv          BENIGN                          189067   
                                                   Bot                               1966   
Monday-WorkingHours.pcap_ISCX.csv                  BENIGN                          529918   
Thursday-WorkingHours-Afternoon-Infilteration.p... BENIGN                          288566   
                                                   Infiltration                        36   
Thursday-WorkingHours-Morning-WebAttacks.pcap_I... BENIGN                          168186   
                                                   Web Attack - Brute Force          1507   
                                                   Web Attack - SQL Injection          21   
                                                   Web Attack - XSS                   652   
Tuesday-WorkingHours.pcap_ISCX.csv                 BENIGN                          432074   
                                                   FTP-Patator                       7938   
                                                   SSH-Patator                       5897   
Wednesday-workingHours.pcap_ISCX.csv               BENIGN                          440031   
                                                   DoS GoldenEye                    10293   
                                                   DoS Hulk                        231073   
                                                   DoS Slowhttptest                  5499   
                                                   DoS slowloris                     5796   
                                                   Heartbleed                          11   

                                                                               Sentinel_Count  \
Source_Dataset                                     Label                                        
Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv   BENIGN                               41771   
                                                   DDoS                                 46528   
Friday-WorkingHours-Afternoon-PortScan.pcap_ISC... BENIGN                               76878   
                                                   PortScan                                64   
Friday-WorkingHours-Morning.pcap_ISCX.csv          BENIGN                              118419   
                                                   Bot                                      0   
Monday-WorkingHours.pcap_ISCX.csv                  BENIGN                              299094   
Thursday-WorkingHours-Afternoon-Infilteration.p... BENIGN                              164348   
                                                   Infiltration                             3   
Thursday-WorkingHours-Morning-WebAttacks.pcap_I... BENIGN                              102356   
                                                   Web Attack - Brute Force                 0   
                                                   Web Attack - SQL Injection               0   
                                                   Web Attack - XSS                        17   
Tuesday-WorkingHours.pcap_ISCX.csv                 BENIGN                              255093   
                                                   FTP-Patator                           1494   
                                                   SSH-Patator                             12   
Wednesday-workingHours.pcap_ISCX.csv               BENIGN                              261022   
   

In [68]:
binary_data = clean_data.copy()

binary_data["Binary_Label"] = np.where(
    binary_data["Label"] == "BENIGN",
    "BENIGN",
    "ATTACK"
)

sentinel_features = [
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

for col in sentinel_features:

    print("=" * 70)
    print(f"Feature: {col}")

    result = binary_data.groupby("Binary_Label")[col].agg(
        Total_Rows="count",
        Sentinel_Count=lambda x: (x == -1).sum()
    )

    result["Sentinel_Percentage"] = (
        result["Sentinel_Count"] / result["Total_Rows"]
    ) * 100

    display(result)

Feature: Init_Win_bytes_forward


,Total_Rows,Sentinel_Count,Sentinel_Percentage
Binary_Label,,,
ATTACK,557646,7,0.001255
BENIGN,2273097,1001182,44.044843


Feature: Init_Win_bytes_backward


,Total_Rows,Sentinel_Count,Sentinel_Percentage
Binary_Label,,,
ATTACK,557646,122571,21.980073
BENIGN,2273097,1318981,58.025724


In [69]:
# 1. Binary Features
binary_features = [
    col for col in numerical_features
    if set(clean_data[col].dropna().unique()).issubset({0, 1})
]

# 2. Rate Features
rate_features = [
    col for col in numerical_features
    if "Rate" in col or "/s" in col
]

# 3. Time Features
time_features = [
    col for col in numerical_features
    if any(word in col for word in ["Duration", "IAT", "Active", "Idle"])
]

# 4. TCP Window Features
window_features = [
    col for col in numerical_features
    if "Init_Win" in col
]

# 5. Remaining Features
known_features = set(
    binary_features +
    rate_features +
    time_features +
    window_features
)

other_features = [
    col for col in numerical_features
    if col not in known_features
]

# Display Results
feature_groups = {
    "Binary": binary_features,
    "Rate": rate_features,
    "Time": time_features,
    "TCP Window": window_features,
    "Other": other_features
}

for group_name, columns in feature_groups.items():

    print("=" * 60)
    print(f"{group_name} Features: {len(columns)}")
    print(columns)

print("\nTotal Features:", sum(len(v) for v in feature_groups.values()))

Binary Features: 8
['Fwd PSH Flags', 'Fwd URG Flags', 'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'ECE Flag Count']
Rate Features: 4
['Flow Bytes/s', 'Flow Packets/s', 'Fwd Packets/s', 'Bwd Packets/s']
Time Features: 23
['Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Active Mean', 'Active Std', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min']
TCP Window Features: 2
['Init_Win_bytes_forward', 'Init_Win_bytes_backward']
Other Features: 28
['Destination Port', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet 

In [70]:
# Create a separate copy for binary classification
binary_data = clean_data.copy()

# Create binary target
binary_data["Target"] = (
    binary_data["Label"] != "BENIGN"
).astype("int8")

# Check target distribution
print("Binary Target Distribution:")

display(
    binary_data["Target"].value_counts()
)

print("\nTarget Percentage:")

display(
    binary_data["Target"]
    .value_counts(normalize=True)
    .mul(100)
)

# Verify mapping
print("\nLabel Mapping Verification:")

display(
    binary_data.groupby("Label")["Target"].first()
)

Binary Target Distribution:


Target
0    2273097
1     557646
Name: count, dtype: int64


Target Percentage:


Target
0    80.300366
1    19.699634
Name: proportion, dtype: float64


Label Mapping Verification:


Label
BENIGN                        0
Bot                           1
DDoS                          1
DoS GoldenEye                 1
DoS Hulk                      1
DoS Slowhttptest              1
DoS slowloris                 1
FTP-Patator                   1
Heartbleed                    1
Infiltration                  1
PortScan                      1
SSH-Patator                   1
Web Attack - Brute Force      1
Web Attack - SQL Injection    1
Web Attack - XSS              1
Name: Target, dtype: int8

In [71]:
# Exclude labels and metadata from feature grouping
feature_cols = [
    col for col in binary_data.columns
    if col not in ["Label", "Source_Dataset", "Target"]
]

# Create a unique group ID for each feature pattern
binary_data["Group_ID"] = pd.util.hash_pandas_object(
    binary_data[feature_cols],
    index=False
)

# Verify number of unique groups
print("Total Rows:", len(binary_data))

print(
    "Unique Feature Groups:",
    binary_data["Group_ID"].nunique()
)

print(
    "Rows belonging to repeated groups:",
    binary_data["Group_ID"].duplicated(keep=False).sum()
)

print(
    "Largest Group Size:",
    binary_data["Group_ID"].value_counts().max()
)

Total Rows: 2830743
Unique Feature Groups: 2521664
Rows belonging to repeated groups: 404558
Largest Group Size: 9329


In [72]:
# Analyze labels inside each feature group

group_analysis = binary_data.groupby("Group_ID").agg(
    Group_Size=("Target", "size"),
    Unique_Labels=("Target", "nunique"),
    Attack_Count=("Target", "sum")
)

group_analysis["Benign_Count"] = (
    group_analysis["Group_Size"]
    - group_analysis["Attack_Count"]
)

print("Groups with one label:")
print(
    (group_analysis["Unique_Labels"] == 1).sum()
)

print("\nGroups with conflicting binary labels:")
print(
    (group_analysis["Unique_Labels"] > 1).sum()
)

print("\nGroup size distribution:")
display(
    group_analysis["Group_Size"].describe()
)

print("\nLargest 10 Groups:")
display(
    group_analysis.sort_values(
        "Group_Size",
        ascending=False
    ).head(10)
)

Groups with one label:
2520966

Groups with conflicting binary labels:
698

Group size distribution:


count    2.521664e+06
mean     1.122569e+00
std      1.083615e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      1.000000e+00
max      9.329000e+03
Name: Group_Size, dtype: float64


Largest 10 Groups:


,Group_Size,Unique_Labels,Attack_Count,Benign_Count
Group_ID,,,,
13745600753527817152,9329,1,9329,0
3387688766074419427,6814,1,6814,0
5575404558521757948,6613,1,6613,0
17921595187844913086,5846,1,5846,0
17031200313272728639,4314,1,4314,0
16322532487507598097,3727,1,0,3727
18443911984605234305,3362,2,3360,2
17995800637875211881,3070,1,0,3070
16521155017463791900,2547,1,2547,0


In [73]:
# Get groups with conflicting binary labels
conflicting_groups = group_analysis[
    group_analysis["Unique_Labels"] > 1
].copy()

print("Number of Conflicting Groups:",
      len(conflicting_groups))

print("\nGroup Size Statistics:")
display(
    conflicting_groups["Group_Size"].describe()
)

print("\nTotal Rows in Conflicting Groups:",
      conflicting_groups["Group_Size"].sum())

print("\nTotal ATTACK Rows:",
      conflicting_groups["Attack_Count"].sum())

print("\nTotal BENIGN Rows:",
      conflicting_groups["Benign_Count"].sum())

print("\nLargest 20 Conflicting Groups:")
display(
    conflicting_groups.sort_values(
        "Group_Size",
        ascending=False
    ).head(20)
)

Number of Conflicting Groups: 698

Group Size Statistics:


count     698.000000
mean       10.057307
std       128.139079
min         2.000000
25%         2.000000
50%         3.000000
75%         4.000000
max      3362.000000
Name: Group_Size, dtype: float64


Total Rows in Conflicting Groups: 7020

Total ATTACK Rows: 6284

Total BENIGN Rows: 736

Largest 20 Conflicting Groups:


,Group_Size,Unique_Labels,Attack_Count,Benign_Count
Group_ID,,,,
18443911984605234305,3362,2,3360,2
2607585622318211399,354,2,351,3
10957786228921203511,164,2,163,1
876583480413491985,133,2,131,2
8815447898753496465,81,2,80,1
7595537112484414330,61,2,60,1
10168274999310684712,54,2,53,1
3334527046142958702,48,2,47,1
1282744280887483168,47,2,45,2


In [74]:

from sklearn.model_selection import GroupShuffleSplit
import numpy as np

# Target and Group IDs
y = binary_data["Target"].to_numpy()
groups = binary_data["Group_ID"].to_numpy()

# First split: 70% Train+Validation, 30% Test
split_1 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_val_idx, test_idx = next(
    split_1.split(
        np.zeros(len(binary_data)),
        y,
        groups
    )
)

print("First split completed!")

# Second split: 70% Train, 30% Validation
# of the remaining 70%
split_2 = GroupShuffleSplit(
    n_splits=1,
    test_size=0.30,
    random_state=42
)

train_relative_idx, val_relative_idx = next(
    split_2.split(
        np.zeros(len(train_val_idx)),
        y[train_val_idx],
        groups[train_val_idx]
    )
)

# Convert relative indices to original indices
train_idx = train_val_idx[train_relative_idx]
val_idx = train_val_idx[val_relative_idx]

print("Second split completed!")

# Keep indices for now to save RAM
print("Train rows:", len(train_idx))
print("Validation rows:", len(val_idx))
print("Test rows:", len(test_idx))


First split completed!
Second split completed!
Train rows: 1388907
Validation rows: 588312
Test rows: 853524


In [75]:

train_groups = set(groups[train_idx])
val_groups = set(groups[val_idx])
test_groups = set(groups[test_idx])

print("Train ∩ Validation:", len(train_groups & val_groups))
print("Train ∩ Test:", len(train_groups & test_groups))
print("Validation ∩ Test:", len(val_groups & test_groups))


Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [76]:

for name, idx in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx)
]:
    print("=" * 50)
    print(name)

    distribution = (
        binary_data.iloc[idx]["Target"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )

    print("BENIGN (0):", round(distribution.get(0, 0), 3), "%")
    print("ATTACK (1):", round(distribution.get(1, 0), 3), "%")


Train
BENIGN (0): 80.342 %
ATTACK (1): 19.658 %
Validation
BENIGN (0): 81.549 %
ATTACK (1): 18.451 %
Test
BENIGN (0): 79.372 %
ATTACK (1): 20.628 %


In [77]:
# Features: exclude metadata and labels
excluded_cols = [
    "Label",
    "Source_Dataset",
    "Target",
    "Group_ID"
]

feature_cols = [
    col for col in binary_data.columns
    if col not in excluded_cols
]

# Extract target
y_train = binary_data.iloc[train_idx]["Target"]
y_val = binary_data.iloc[val_idx]["Target"]
y_test = binary_data.iloc[test_idx]["Target"]

print("Number of Features:", len(feature_cols))
print("Training Samples:", len(y_train))
print("Validation Samples:", len(y_val))
print("Testing Samples:", len(y_test))

Number of Features: 65
Training Samples: 1388907
Validation Samples: 588312
Testing Samples: 853524


In [78]:

X_train = binary_data.iloc[train_idx][feature_cols]
X_val = binary_data.iloc[val_idx][feature_cols]
X_test = binary_data.iloc[test_idx][feature_cols]

print("Missing Values in Train:", X_train.isna().sum().sum())
print("Missing Values in Validation:", X_val.isna().sum().sum())
print("Missing Values in Test:", X_test.isna().sum().sum())

print("\nFeatures with Missing Values in Train:")
display(
    X_train.isna().sum()
    .loc[lambda x: x > 0]
    .sort_values(ascending=False)
)


Missing Values in Train: 5465
Missing Values in Validation: 1661
Missing Values in Test: 2153

Features with Missing Values in Train:


Flow Packets/s          1934
Flow Bytes/s            1920
Flow IAT Min            1409
Flow Duration             48
Flow IAT Mean             48
Flow IAT Max              48
Fwd Header Length         20
min_seg_size_forward      20
Bwd Header Length         12
Fwd IAT Min                6
dtype: int64

In [79]:

from sklearn.impute import SimpleImputer
import joblib
import os

# Create imputer
imputer = SimpleImputer(strategy="median")

# Fit only on training data
X_train_imputed = imputer.fit_transform(X_train)

# Apply the same medians to validation and test
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

# Save the fitted imputer
os.makedirs("../Models", exist_ok=True)

joblib.dump(
    imputer,
    "../Models/median_imputer.pkl"
)

print("Imputer saved successfully!")

print("Train Shape:", X_train_imputed.shape)
print("Validation Shape:", X_val_imputed.shape)
print("Test Shape:", X_test_imputed.shape)
print("Number of Features:", len(imputer.feature_names_in_))


Imputer saved successfully!
Train Shape: (1388907, 65)
Validation Shape: (588312, 65)
Test Shape: (853524, 65)
Number of Features: 65


In [80]:

print("Train Missing:", np.isnan(X_train_imputed).sum())
print("Validation Missing:", np.isnan(X_val_imputed).sum())
print("Test Missing:", np.isnan(X_test_imputed).sum())


Train Missing: 0
Validation Missing: 0
Test Missing: 0


In [81]:

# Convert training data back to a DataFrame
X_train_df = pd.DataFrame(
    X_train_imputed,
    columns=feature_cols,
    index=binary_data.index[train_idx]
)

# Check selected features
selected_features = [
    "Destination Port",
    "Flow Duration",
    "Total Fwd Packets",
    "Total Length of Fwd Packets",
    "Flow Bytes/s",
    "Flow Packets/s",
    "Idle Mean"
]

display(
    X_train_df[selected_features].describe().T
)


,count,mean,std,min,25%,50%,75%,max
Destination Port,1388907.0,8.060424e+03,1.827462e+04,0.000000,53.000000,80.000000,4.430000e+02,6.553400e+04
Flow Duration,1388907.0,1.476022e+07,3.363759e+07,0.000000,154.000000,31291.000000,3.172256e+06,1.200000e+08
Total Fwd Packets,1388907.0,9.666863e+00,7.982035e+02,1.000000,2.000000,2.000000,5.000000e+00,2.177970e+05
Total Length of Fwd Packets,1388907.0,5.399209e+02,5.640193e+03,0.000000,12.000000,62.000000,1.870000e+02,1.288022e+06
Flow Bytes/s,1388907.0,1.499558e+06,2.549362e+07,0.000000,119.437141,4602.991945,1.666667e+05,2.071000e+09
Flow Packets/s,1388907.0,7.291570e+04,2.582948e+05,0.016701,3.495916,110.332653,2.325581e+04,4.000000e+06
Idle Mean,1388907.0,8.287185e+06,2.358134e+07,0.000000,0.000000,0.000000,0.000000e+00,1.200000e+08


In [82]:

skewness = X_train_df.skew().sort_values(
    ascending=False
)

skewness_summary = pd.DataFrame({
    "Skewness": skewness,
    "Absolute_Skewness": skewness.abs()
})

display(skewness_summary.head(20))


,Skewness,Absolute_Skewness
act_data_pkt_fwd,271.052634,271.052634
Total Backward Packets,233.602377,233.602377
Subflow Bwd Bytes,233.299519,233.299519
Total Length of Bwd Packets,233.298946,233.298946
Total Fwd Packets,233.042599,233.042599
Bwd Header Length,232.553444,232.553444
Fwd Header Length,232.443802,232.443802
Subflow Fwd Bytes,133.572458,133.572458
Total Length of Fwd Packets,133.572458,133.572458
Fwd URG Flags,99.945777,99.945777


In [83]:

# Features that should not receive log1p automatically
excluded_log_features = [
    "Fwd PSH Flags",
    "Fwd URG Flags",
    "FIN Flag Count",
    "RST Flag Count",
    "PSH Flag Count",
    "ACK Flag Count",
    "URG Flag Count",
    "ECE Flag Count",
    "Destination Port",
    "Init_Win_bytes_forward",
    "Init_Win_bytes_backward"
]

# Select highly positively skewed, non-negative features
candidate_log_features = [
    col for col in feature_cols
    if (
        skewness[col] > 1
        and col not in excluded_log_features
        and X_train_df[col].min() >= 0
    )
]

print("Number of candidate features:", len(candidate_log_features))
print("\nCandidate Features:")

for col in candidate_log_features:
    print(col)

# Check zero percentages for these features
zero_percentage = (
    X_train_df[candidate_log_features]
    .eq(0)
    .mean() * 100
).sort_values(ascending=False)

print("\nZero Percentage:")
display(zero_percentage.to_frame("Zero_Percentage"))


Number of candidate features: 53

Candidate Features:
Flow Duration
Total Fwd Packets
Total Backward Packets
Total Length of Fwd Packets
Total Length of Bwd Packets
Fwd Packet Length Max
Fwd Packet Length Min
Fwd Packet Length Mean
Fwd Packet Length Std
Bwd Packet Length Max
Bwd Packet Length Min
Bwd Packet Length Mean
Bwd Packet Length Std
Flow Bytes/s
Flow Packets/s
Flow IAT Mean
Flow IAT Std
Flow IAT Max
Flow IAT Min
Fwd IAT Total
Fwd IAT Mean
Fwd IAT Std
Fwd IAT Max
Fwd IAT Min
Bwd IAT Total
Bwd IAT Mean
Bwd IAT Std
Bwd IAT Max
Bwd IAT Min
Fwd Header Length
Bwd Header Length
Fwd Packets/s
Bwd Packets/s
Min Packet Length
Max Packet Length
Packet Length Mean
Packet Length Std
Packet Length Variance
Down/Up Ratio
Average Packet Size
Avg Fwd Segment Size
Avg Bwd Segment Size
Subflow Fwd Bytes
Subflow Bwd Bytes
act_data_pkt_fwd
Active Mean
Active Std
Active Max
Active Min
Idle Mean
Idle Std
Idle Max
Idle Min

Zero Percentage:


,Zero_Percentage
Active Std,92.744871
Idle Std,91.899530
Active Max,80.291049
Active Mean,80.291049
Active Min,80.291049
Idle Mean,79.990021
Idle Min,79.990021
Idle Max,79.990021
Bwd Packet Length Std,71.618618
Bwd IAT Std,71.360861


In [84]:

# Create separate DataFrames for all splits
X_val_df = pd.DataFrame(
    X_val_imputed,
    columns=feature_cols,
    index=binary_data.index[val_idx]
)

X_test_df = pd.DataFrame(
    X_test_imputed,
    columns=feature_cols,
    index=binary_data.index[test_idx]
)

# Preserve the original imputed data
X_train_log_df = X_train_df.copy()
X_val_log_df = X_val_df.copy()
X_test_log_df = X_test_df.copy()

# Apply log1p to candidate features
X_train_log_df[candidate_log_features] = np.log1p(
    X_train_log_df[candidate_log_features]
)

X_val_log_df[candidate_log_features] = np.log1p(
    X_val_log_df[candidate_log_features]
)

X_test_log_df[candidate_log_features] = np.log1p(
    X_test_log_df[candidate_log_features]
)

print("Number of transformed features:", len(candidate_log_features))
print("Train Shape:", X_train_log_df.shape)
print("Validation Shape:", X_val_log_df.shape)
print("Test Shape:", X_test_log_df.shape)


Number of transformed features: 53
Train Shape: (1388907, 65)
Validation Shape: (588312, 65)
Test Shape: (853524, 65)


In [85]:

skew_comparison = pd.DataFrame({
    "Before": X_train_df[candidate_log_features].skew(),
    "After": X_train_log_df[candidate_log_features].skew()
})

skew_comparison["Improvement"] = (
    skew_comparison["Before"].abs()
    - skew_comparison["After"].abs()
)

display(
    skew_comparison.sort_values(
        "Improvement",
        ascending=False
    )
)


,Before,After,Improvement
act_data_pkt_fwd,271.052634,1.475576,269.577058
Subflow Bwd Bytes,233.299519,0.147207,233.152311
Total Length of Bwd Packets,233.298946,0.147207,233.151738
Total Backward Packets,233.602377,1.386764,232.215612
Bwd Header Length,232.553444,-0.583676,231.969767
Fwd Header Length,232.443802,1.143026,231.300776
Total Fwd Packets,233.042599,1.776324,231.266276
Total Length of Fwd Packets,133.572458,-0.081689,133.490768
Subflow Fwd Bytes,133.572458,-0.081689,133.490768
Flow Bytes/s,45.593298,-0.187315,45.405983


In [86]:
print("Train:", X_train_df.shape)
print("Validation:", X_val_df.shape)
print("Test:", X_test_df.shape)

print("\nLog Train:", X_train_log_df.shape)
print("Log Validation:", X_val_log_df.shape)
print("Log Test:", X_test_log_df.shape)

print("\nMissing Values:")
print("Train:", X_train_df.isna().sum().sum())
print("Validation:", X_val_df.isna().sum().sum())
print("Test:", X_test_df.isna().sum().sum())

print("\nTarget Distribution:")
print(y_train.value_counts(normalize=True) * 100)

Train: (1388907, 65)
Validation: (588312, 65)
Test: (853524, 65)

Log Train: (1388907, 65)
Log Validation: (588312, 65)
Log Test: (853524, 65)

Missing Values:
Train: 0
Validation: 0
Test: 0

Target Distribution:
Target
0    80.341664
1    19.658336
Name: proportion, dtype: float64


In [87]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [88]:
import os

PROCESSED_PATH = "../Dataset/processed"

os.makedirs(PROCESSED_PATH, exist_ok=True)

print("Processed directory:", os.path.abspath(PROCESSED_PATH))
print("Directory exists:", os.path.exists(PROCESSED_PATH))

Processed directory: e:\Projects\Sentinel AI\Dataset\processed
Directory exists: True


In [89]:
# Add target column to training features
X_train_df["Target"] = y_train.to_numpy()

# Save as Parquet
train_path = os.path.join(
    PROCESSED_PATH,
    "train_baseline.parquet"
)

X_train_df.to_parquet(
    train_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Training data saved successfully!")
print("Path:", os.path.abspath(train_path))
print("Shape:", X_train_df.shape)
print("File size (MB):", round(os.path.getsize(train_path) / (1024**2), 2))

Training data saved successfully!
Path: e:\Projects\Sentinel AI\Dataset\processed\train_baseline.parquet
Shape: (1388907, 66)
File size (MB): 159.37


In [90]:
# Add target column to validation features
X_val_df["Target"] = y_val.to_numpy()

# Save as Parquet
val_path = os.path.join(
    PROCESSED_PATH,
    "validation_baseline.parquet"
)

X_val_df.to_parquet(
    val_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Validation data saved successfully!")
print("Path:", os.path.abspath(val_path))
print("Shape:", X_val_df.shape)
print("File size (MB):", round(os.path.getsize(val_path) / (1024**2), 2))

Validation data saved successfully!
Path: e:\Projects\Sentinel AI\Dataset\processed\validation_baseline.parquet
Shape: (588312, 66)
File size (MB): 69.05


In [91]:
# Add target column to test features
X_test_df["Target"] = y_test.to_numpy()

# Save as Parquet
test_path = os.path.join(
    PROCESSED_PATH,
    "test_baseline.parquet"
)

X_test_df.to_parquet(
    test_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Test data saved successfully!")
print("Path:", os.path.abspath(test_path))
print("Shape:", X_test_df.shape)
print("File size (MB):", round(os.path.getsize(test_path) / (1024**2), 2))

Test data saved successfully!
Path: e:\Projects\Sentinel AI\Dataset\processed\test_baseline.parquet
Shape: (853524, 66)
File size (MB): 98.05


In [92]:
X_train_log_df["Target"] = y_train.to_numpy()

train_log_path = os.path.join(
    PROCESSED_PATH,
    "train_log.parquet"
)

X_train_log_df.to_parquet(
    train_log_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Log Training data saved!")
print("Shape:", X_train_log_df.shape)
print("File size (MB):", round(os.path.getsize(train_log_path) / (1024**2), 2))

Log Training data saved!
Shape: (1388907, 66)
File size (MB): 173.56


In [93]:
X_val_log_df["Target"] = y_val.to_numpy()

val_log_path = os.path.join(
    PROCESSED_PATH,
    "validation_log.parquet"
)

X_val_log_df.to_parquet(
    val_log_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Log Validation data saved!")
print("Shape:", X_val_log_df.shape)
print("File size (MB):", round(os.path.getsize(val_log_path) / (1024**2), 2))

Log Validation data saved!
Shape: (588312, 66)
File size (MB): 74.8


In [94]:
X_test_log_df["Target"] = y_test.to_numpy()

test_log_path = os.path.join(
    PROCESSED_PATH,
    "test_log.parquet"
)

X_test_log_df.to_parquet(
    test_log_path,
    index=False,
    engine="pyarrow",
    compression="snappy"
)

print("Log Test data saved!")
print("Shape:", X_test_log_df.shape)
print("File size (MB):", round(os.path.getsize(test_log_path) / (1024**2), 2))

Log Test data saved!
Shape: (853524, 66)
File size (MB): 106.48


In [ ]:

saved_files = [
    "train_baseline.parquet",
    "validation_baseline.parquet",
    "test_baseline.parquet",
    "train_log.parquet",
    "validation_log.parquet",
    "test_log.parquet"
]

for file_name in saved_files:
    file_path = os.path.join(PROCESSED_PATH, file_name)

    df_check = pd.read_parquet(file_path)

    print("=" * 60)
    print("File:", file_name)
    print("Shape:", df_check.shape)
    print("Missing Values:", df_check.isna().sum().sum())
    print("Target Distribution:")
    print(df_check["Target"].value_counts(normalize=True).mul(100).round(2))


File: train_baseline.parquet
Shape: (1388907, 66)
Missing Values: 0
Target Distribution:
Target
0    80.34
1    19.66
Name: proportion, dtype: float64
File: validation_baseline.parquet
Shape: (588312, 66)
Missing Values: 0
Target Distribution:
Target
0    81.55
1    18.45
Name: proportion, dtype: float64
File: test_baseline.parquet
Shape: (853524, 66)
Missing Values: 0
Target Distribution:
Target
0    79.37
1    20.63
Name: proportion, dtype: float64


In [ ]:
[name for name in globals() if "imput" in name.lower()]

[]